# Session 3: exploratory data analysis of your dataset

Session 2 focused on finding and loading one media file. This session moves outward: a metadata table describes a collection of samples and helps you see how that collection is organized.

Part 1 is a short worked camera-trap example. **Part 2, using your own metadata, is the main activity.** If your metadata are not ready, use the example table in Part 2 and record what you need before January.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Part 1: worked camera-trap example

The supplied table is fictional, but its columns resemble common camera-trap and iWildCam metadata: an ID and filename for each record, a category, a location, a deployment or sequence-like group, a timestamp, and annotation status. It is small enough to inspect directly.

In [ ]:
possible_example_paths = [
    Path.cwd() / "example_metadata.csv",
    Path.cwd() / "03_understand_your_data" / "example_metadata.csv",
    Path.cwd() / "Deliverables" / "prep" / "03_understand_your_data" / "example_metadata.csv",
]
example_path = next((path for path in possible_example_paths if path.exists()), None)
if example_path is None:
    raise FileNotFoundError(f"Could not find example_metadata.csv from {Path.cwd()}")

example = pd.read_csv(example_path)
print("Rows and columns:", example.shape)
display(example.head())

## Read the structure before calculating summaries

In this table, one row represents one image record. `sample_id` identifies the record and `filename` points toward its media. Other columns describe that record or connect it to a larger group.

The next cells answer three basic EDA questions:

1. What information is available?
2. What important information is missing?
3. How are records distributed across a target and a natural group?

In [ ]:
print("Columns:", example.columns.tolist())
print("\nData types:")
display(example.dtypes.to_frame("dtype"))
print("Missing values:")
display(example.isna().sum().to_frame("missing_rows"))

In [ ]:
species_counts = example["species"].fillna("<missing>").value_counts()
site_counts = example["site"].fillna("<missing>").value_counts()
species_by_site = pd.crosstab(
    example["site"],
    example["species"].fillna("<missing>"),
)
print("Species counts:")
display(species_counts.to_frame("records"))
print("Species by site:")
display(species_by_site)

In [ ]:
ax = species_counts.plot(kind="bar", color="seagreen")
ax.set(title="Example records by species", xlabel="Species", ylabel="Records")
plt.tight_layout()
plt.show()

A useful EDA observation names evidence rather than merely saying a graph is interesting. For example: **four records lack a species label, with one unlabeled record at each site.** A useful follow-up question is: **does `annotation_status` explain those missing labels?**

# Part 2: explore your own metadata

Set `MY_DATA_PATH` to a small CSV extract from your project. Keep research data outside this repository. If your metadata are not ready, leave it as `None`; the notebook will use the example so you can still practice the workflow.

In [ ]:
MY_DATA_PATH = None  # Example: Path("/path/to/my_small_metadata.csv")

if MY_DATA_PATH is None:
    metadata = example.copy()
    using_example = True
    print("Using the supplied example. Replace MY_DATA_PATH when your metadata are ready.")
else:
    metadata = pd.read_csv(Path(MY_DATA_PATH).expanduser())
    using_example = False
    print("Using your metadata:", Path(MY_DATA_PATH).expanduser())

## 1. Define what the table represents

Do not begin with calculations. First determine what the table means.

- **One row represents:** _Write here._
- **The table covers:** _Write here._
- **It was created by or exported from:** _Write here._
- **Important information stored elsewhere:** _Write here._

## 2. Inspect shape, columns, and types

Run this cell, then inspect several rows in the CSV itself. Do not assume Pandas interpreted every column correctly—dates and identifiers often need special attention.

In [ ]:
print("Rows and columns:", metadata.shape)
print("Columns:", metadata.columns.tolist())
display(metadata.head())
display(metadata.dtypes.to_frame("dtype"))

- **A column whose type needs checking:** _Write here, or state that none stood out._
- **Why:** _Write here._

## 3. Name the columns that organize your data

Edit these settings to match your table. A natural group contains related observations that should not be treated as completely independent—for example a site, deployment, sequence, individual, video, recorder, or sampling event. Use `None` if your table truly has no corresponding column.

In [ ]:
ID_COLUMN = "sample_id"
MEDIA_COLUMN = "filename"
TARGET_COLUMN = "species"
GROUP_COLUMN = "site"

selected_columns = {
    "ID_COLUMN": ID_COLUMN,
    "MEDIA_COLUMN": MEDIA_COLUMN,
    "TARGET_COLUMN": TARGET_COLUMN,
    "GROUP_COLUMN": GROUP_COLUMN,
}
for role, column in selected_columns.items():
    if column is not None and column not in metadata.columns:
        raise KeyError(f"{role} is set to {column!r}, but that column is not in the table")
display(pd.DataFrame([selected_columns]))

- **Why is this the appropriate ID column?** _Write here._
- **What does the natural group represent?** _Write here._
- **If a needed role is missing, where is that information stored?** _Write here._

## 4. Check missing values and identifiers

Missing values are not automatically errors. Identify where they occur before deciding what they mean. IDs require a separate check: duplicated IDs may be expected if one sample occupies several rows, or may reveal a problem if every row should be unique.

In [ ]:
missing_summary = metadata.isna().sum().sort_values(ascending=False)
display(missing_summary.to_frame("missing_rows"))

if ID_COLUMN is not None:
    print("Rows:", len(metadata))
    print("Unique non-missing IDs:", metadata[ID_COLUMN].nunique())
    print("Rows with duplicated IDs:", metadata[ID_COLUMN].duplicated(keep=False).sum())

- **Most consequential missing value:** _Name the column and explain why it matters._
- **Are repeated IDs expected for what one row represents?** _Write here._

## 5. Count a target and a natural group

The helper below includes missing values explicitly. Run it for the target and group you selected. If either setting is `None`, explain what alternative summary would be meaningful for your project.

In [ ]:
def counts_including_missing(table, column):
    if column is None:
        return None
    return table[column].fillna("<missing>").value_counts()

target_counts = counts_including_missing(metadata, TARGET_COLUMN)
group_counts = counts_including_missing(metadata, GROUP_COLUMN)
if target_counts is not None:
    print("Target counts:")
    display(target_counts.to_frame("rows"))
if group_counts is not None:
    print("Group counts:")
    display(group_counts.to_frame("rows"))

- **A notable target count:** _Write here._
- **A notable group count:** _Write here._
- **What these counts do not tell you:** _Write here._

## 6. Compare the target across groups

A total count can hide differences among sites, deployments, sequences, or other groups. If you selected both columns, the crosstab below makes that relationship visible.

In [ ]:
if TARGET_COLUMN is not None and GROUP_COLUMN is not None:
    target_by_group = pd.crosstab(
        metadata[GROUP_COLUMN].fillna("<missing>"),
        metadata[TARGET_COLUMN].fillna("<missing>"),
    )
    display(target_by_group)
else:
    print("Choose both TARGET_COLUMN and GROUP_COLUMN to make this comparison.")

- **One relationship visible here but hidden in the totals:** _Write here._

## 7. Make one useful plot

Choose whether the target or group counts better address a question about your data. Assign that Series and a clear title below. Avoid plotting an identifier column: a separate bar for every sample rarely helps.

In [ ]:
counts_to_plot = target_counts  # Change to group_counts if that is more useful.
plot_title = f"Rows by {TARGET_COLUMN}"  # Edit to describe your plot.

if counts_to_plot is None:
    raise ValueError("Choose a column and create counts before plotting")
ax = counts_to_plot.plot(kind="bar", color="steelblue")
ax.set(title=plot_title, xlabel=counts_to_plot.index.name, ylabel="Rows")
plt.tight_layout()
plt.show()

- **What the plot shows:** _State one observation supported by the plot._
- **Question it raises:** _State one concrete question to discuss with an instructor._

## 8. Check links from metadata to media when possible

This connects session 3 back to the file work in session 2 and forward to the dataset object in session 4. If you have a small local media directory and a filename column, set `MY_MEDIA_DIR`. Otherwise leave it as `None` and describe how media paths are recorded in your project.

In [ ]:
MY_MEDIA_DIR = None  # Example: Path("/path/to/my_small_media_sample")

if MY_MEDIA_DIR is not None and MEDIA_COLUMN is not None:
    media_dir = Path(MY_MEDIA_DIR).expanduser()
    expected_paths = metadata[MEDIA_COLUMN].dropna().map(lambda name: media_dir / str(name))
    missing_media = expected_paths[~expected_paths.map(Path.exists)]
    print("Media paths checked:", len(expected_paths))
    print("Files not found:", len(missing_media))
    display(missing_media.head().to_frame("missing_path"))
else:
    print("Media check skipped. Record below how table rows connect to media.")

- **How a row connects to its media:** _Write here._
- **How a row connects to annotations, if separate:** _Write here._
- **Any mismatches or information still needed:** _Write here._

## 9. Inspect a focused view of the table

Filtering is useful during EDA when a summary raises a question. Write one Boolean condition that selects rows you want to inspect—for example missing labels, one site, one annotation status, or records with a quality note. This is an exploratory view, **not** your final development or model-training subset.

In [ ]:
# Replace this condition with one that addresses your EDA question.
if TARGET_COLUMN is not None:
    condition = metadata[TARGET_COLUMN].isna()
else:
    condition = pd.Series(True, index=metadata.index)
focused_view = metadata.loc[condition].copy()
print("Selected rows:", len(focused_view))
display(focused_view.head(10))

- **Why I selected these rows:** _Write here._
- **What I learned by inspecting them:** _Write here._

# EDA notes to bring to the instructors

Complete these in plain language. The goal is not a polished report.

- **I used my own metadata / the supplied fallback:** _Choose one._
- **Approximate scope represented by this table:** _Write here._
- **One row represents:** _Write here._
- **Sample ID and media connection:** _Write here._
- **Most important missing or questionable information:** _Write here._
- **Most useful count or plot:** _Write here._
- **One concrete question for the instructors:** _Write here._
- **If I used the fallback, what is needed to explore my data:** _Write here._

## Connection to the next sessions

In session 4, you will use IDs and paths to load the files belonging to one complete sample through a dataset class. In session 6, you and the instructors will use observations from this EDA to plan a manageable working subset. Formal train/validation/test splitting remains course content.